# 01 — Data Preparation
Load BTC/USDT 15m OHLCV from local CSV (Binance klines format),
resample to 1H and 4H, clean, and save to parquet on Google Drive.

In [ ]:
# Install dependencies
# torch, numpy, pandas are pre-installed by Colab — do NOT reinstall them.
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tensorboard tqdm pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# Clone/update repo from GitHub (local Colab filesystem — fast)
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/sergul74/Scalp2.git {REPO_DIR}

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2', '__init__.py')):
    raise FileNotFoundError('scalp2 package not found after clone!')

sys.path.insert(0, REPO_DIR)
print(f'Using project at {REPO_DIR}')

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

from scalp2.config import load_config
config = load_config(f'{REPO_DIR}/config.yaml')

# Output dirs on Google Drive for persistence
config.data.processed_dir = '/content/drive/MyDrive/scalp2/data/processed'
os.makedirs(config.data.processed_dir, exist_ok=True)

print(f'Symbol: {config.data.symbol}')
print(f'Date range: {config.data.date_range.start} to {config.data.date_range.end}')
print(f'Timeframes: {config.data.timeframes.primary}, {config.data.timeframes.mtf}')

In [ ]:
# Fetch new data via ccxt to update the CSV (if needed)
import ccxt
import pandas as pd
from datetime import datetime, timezone
import time
import os

CSV_PATH = '/content/drive/MyDrive/scalp2/data/btcusdt_15min.csv'
symbol = config.data.symbol
timeframe = config.data.timeframes.primary

print(f"Checking for new data for {symbol}...")

# Use Binance (spot) if USDM is blocked in Colab's region, or proxy
try:
    exchange = ccxt.binanceusdm({'enableRateLimit': True})
    exchange.fetch_time() # Test connection
except Exception as e:
    print(f"USDM blocked ({e}). Switching to spot exchange as fallback.")
    exchange = ccxt.binance({'enableRateLimit': True})

# 1. Load existing data if any
df_old = None
if os.path.exists(CSV_PATH):
    # Read without index first to find the time column
    df_temp = pd.read_csv(CSV_PATH, nrows=5)
    
    # Identify time column (usually 'timestamp', 'date', 'time', or first unnamed col)
    time_col = None
    for col in df_temp.columns:
        if 'time' in col.lower() or 'date' in col.lower():
            time_col = col
            break
            
    if not time_col:
        time_col = df_temp.columns[0] # Fallback to first column
        
    print(f"Using column '{time_col}' for timestamp.")
    
    # Read full CSV
    df_old = pd.read_csv(CSV_PATH)
    
    # Convert properly
    try:
        # Convert ms timestamps or string dates to tz-aware datetime
        if pd.api.types.is_numeric_dtype(df_old[time_col]):
            if df_old[time_col].max() > 1e11: # it's in ms
                df_old[time_col] = pd.to_datetime(df_old[time_col], unit='ms', utc=True)
            else: # it's in seconds
                df_old[time_col] = pd.to_datetime(df_old[time_col], unit='s', utc=True)
        else:
            df_old[time_col] = pd.to_datetime(df_old[time_col], utc=True)
            
        df_old.set_index(time_col, inplace=True)
        df_old.index.name = 'timestamp'
        
        last_dt = df_old.index[-1]
        
        # Guard against 1970 date (bad parsing)
        if last_dt.year < 2017:
             print(f"WARNING: Bad date parsed ({last_dt}). Will fetch from config start date.")
             df_old = None
        else:
            since_ts = int(last_dt.timestamp() * 1000) + 1
            print(f"Found existing data ending at {last_dt}. Fetching from here...")
            
    except Exception as e:
        print(f"Error parsing dates: {e}. Will fetch from config start date.")
        df_old = None

if df_old is None:
    start_date = pd.to_datetime(config.data.date_range.start).replace(tzinfo=timezone.utc)
    since_ts = int(start_date.timestamp() * 1000)
    print(f"Fetching from start date {start_date}...")

# Determine end timestamp from config
end_date = pd.to_datetime(config.data.date_range.end).replace(tzinfo=timezone.utc)
end_ts = int(end_date.timestamp() * 1000)

new_data = []
limit = 1000

if since_ts < end_ts:
    print("Downloading new data...")
    while since_ts < end_ts:
        try:
            ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since=since_ts, limit=limit)
            if not ohlcv:
                break
            
            new_data.extend(ohlcv)
            
            last_fetched = ohlcv[-1][0]
            if last_fetched <= since_ts:
                 break # Prevents infinite loop if API returns same timestamp
            since_ts = last_fetched + 1
            
            # Convert ts to readable date for printing
            last_dt_str = datetime.fromtimestamp(last_fetched/1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
            print(f"  Fetched up to {last_dt_str} ", end='\r')
            time.sleep(exchange.rateLimit / 1000) # Respect rate limits
            
        except Exception as e:
            print(f"\nError fetching data: {e}")
            break
    print("\nDownload complete.")
else:
    print("Data is already up to date with config end date.")

# 2. Process and append new data
if new_data:
    columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    df_new = pd.DataFrame([x[:6] for x in new_data], columns=columns)
    df_new['timestamp'] = pd.to_datetime(df_new['timestamp'], unit='ms', utc=True)
    df_new.set_index('timestamp', inplace=True)
    
    if df_old is not None:
        # Combine and drop duplicates
        df_combined = pd.concat([df_old, df_new])
        df_combined = df_combined[~df_combined.index.duplicated(keep='last')].sort_index()
    else:
        df_combined = df_new
        
    # Ensure directory exists
    os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
    
    print(f"Saving updated dataset ({len(df_combined)} rows) to CSV...")
    df_combined.to_csv(CSV_PATH)
    print("Done!")
else:
    print("No new data added.")

In [ ]:
from scalp2.data.preprocessing import load_binance_csv, resample_ohlcv

# Load the 15m CSV from Google Drive
CSV_PATH = '/content/drive/MyDrive/scalp2/data/btcusdt_15min.csv'
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f'CSV not found at {CSV_PATH}\n'
        'Upload btcusdt_15min.csv to: My Drive -> scalp2 -> data/'
    )

df_15m = load_binance_csv(CSV_PATH)

print(f'15m: {len(df_15m):,} bars')
print(f'Columns: {list(df_15m.columns)}')
print(f'Range: {df_15m.index[0]} → {df_15m.index[-1]}')
df_15m.head()

In [ ]:
# Resample to 1H and 4H from 15m
df_1h = resample_ohlcv(df_15m, '1h')
df_4h = resample_ohlcv(df_15m, '4h')

print(f'1h: {len(df_1h):,} bars')
print(f'4h: {len(df_4h):,} bars')

In [ ]:
# Clean all timeframes (gap-fill, validate, optimize dtypes)
from scalp2.data.preprocessing import clean_ohlcv

data = {'15m': df_15m, '1h': df_1h, '4h': df_4h}

for tf in data:
    data[tf] = clean_ohlcv(data[tf], tf)
    print(f'{tf} after cleaning: {len(data[tf]):,} bars')

In [ ]:
# Save cleaned data to Google Drive
for tf, df in data.items():
    path = f'{config.data.processed_dir}/BTC_USDT_{tf}_clean.parquet'
    df.to_parquet(path)
    print(f'Saved {path}  ({len(df):,} bars)')

print('\nData preparation complete.')